In [0]:
WITH raw_data AS (SELECT *
FROM banking_lakehouse.bronze.raw_transactions),

cleaned AS 
(SELECT
    CAST(step AS INT) AS step_hour,
    CAST(type AS STRING) AS transaction_type,
    CAST(amount AS DOUBLE) AS transaction_amount,
        
    CAST(nameOrig AS STRING) AS sender_account_id,
    CAST(oldbalanceOrg AS DOUBLE) AS sender_old_balance,
    CAST(newbalanceOrig AS DOUBLE) AS sender_new_balance,
        
    CAST(nameDest AS STRING) AS receiver_account_id,
    CAST(oldbalanceDest AS DOUBLE) AS receiver_old_balance,
    CAST(newbalanceDest AS DOUBLE) AS receiver_new_balance,
        
    CAST(isFraud AS BOOLEAN) AS is_fraud,
    CAST(isFlaggedFraud AS BOOLEAN) AS is_flagged_fraud
FROM raw_data)


SELECT 
    md5(concat(CAST(step_hour AS STRING), sender_account_id, receiver_account_id, CAST(transaction_amount AS STRING))) AS transaction_id,
    date_add(hour, CAST(step_hour AS INT), '2024-01-01 00:00:00'::timestamp) AS transaction_timestamp,
    *,
    (CASE WHEN receiver_account_id LIKE 'M%' THEN TRUE
    ELSE FALSE
    END) AS is_receiver_merchant,
    ROUND((sender_new_balance + transaction_amount) - sender_old_balance,2) AS sender_balance_error,
    (CASE
      WHEN receiver_account_id LIKE 'M%' THEN 0
      ELSE ROUND((receiver_old_balance + transaction_amount) - sender_new_balance,2)
      END)receiver_balance_error,
    (CASE
      WHEN sender_old_balance = sender_new_balance AND transaction_amount > 0 THEN TRUE ELSE FALSE END) AS is_zero_impact_txn,
      CASE 
        WHEN transaction_amount > 10000 THEN 'HIGH_RISK'
        ELSE 'LOW_RISK'
      END AS risk_level
FROM cleaned;


SELECT *
FROM banking_lakehouse.silver.stg_transactions
where not (transaction_amount > 0)






SELECT * 
FROM banking_lakehouse.sliver.stg_transactions
LIMIT 10

CREATE SCHEMA IF NOT EXISTS banking_lakehouse.silver;

CREATE SCHEMA IF NOT EXISTS banking_lakehouse.gold;

USE SCHEMA silver;

USE SCHEMA gold;


DROP TABLE IF EXISTS silver.customer_scd2


DROP SCHEMA IF EXISTS banking_lakehouse.snapshot CASCADE


SELECT *,
FROM gold.dim_customers
WHERE risk_level = 'LOW_RISK'


SELECT *
FROM silver.customer_scd2


SELECT 
  f.sender_account_id,
  f.transaction_amount,
  f.is_fraud,
  c.risk_level
FROM gold.fct_transaction f
JOIN gold.dim_customers c
ON f.sender_customer_sk = c.customer_sk
WHERE c.risk_level = 'HIGH_RISK'
AND is_fraud = TRUE




SELECT 
  f.sender_account_id,
  f.transaction_amount,
  COUNT(*) as cnt
FROM gold.fct_transaction f
JOIN gold.dim_customers c
ON f.sender_customer_sk = c.customer_sk
GROUP BY 1,2


SELECT *
FROM banking_lakehouse.silver.stg_transactions
ORDER BY step_hour DESC
LIMIT 10


DROP TABLE IF EXISTS stg_transactions


